# Mantenimiento y Gobierno de Datos en Delta Lake

Este notebook detalla el ciclo de vida de optimización de una tabla tras una ingesta incremental. Cuando operamos con múltiples escrituras de tipo append, nos enfrentamos a la fragmentación de metadatos y al fenómeno de Small Files, lo cual degrada el rendimiento de Spark debido al overhead en el driver y al exceso de operaciones I/O.

## Capa de Optimización de Almacenamiento

#### 1. Diagnóstico y Estructura
* **DESCRIBE DETAIL**: Expone metadatos técnicos como el número de archivos, tamaño de la tabla y formato, permitiendo identificar problemas de fragmentación.
* **DESCRIBE HISTORY**: Registra el linaje de operaciones realizadas, facilitando la auditoría de cambios y la identificación de versiones para recuperación.

#### 2. Compactación
* **OPTIMIZE**: Consolida archivos pequeños en archivos de mayor tamaño para minimizar el overhead de metadatos y optimizar las operaciones de lectura.
* **ZORDER BY**: Organiza físicamente los datos basándose en columnas de alta cardinalidad para maximizar la eficiencia del Data Skipping en las consultas.

#### 3. Higiene de Almacenamiento
* **VACUUM**: Elimina físicamente archivos obsoletos que ya no son referenciados por el log de transacciones, liberando espacio y reduciendo costos. Por defecto utiliza la retencion de 7 dias, para cambiarlo se debe ejecutar antes `SET spark.databricks.delta.retentionDurationCheck.enabled = false`

## Gobernanza y Recuperación (Time Travel)

Delta Lake permite acceder a estados anteriores de la tabla, garantizando la integridad de los datos ante errores operativos.

#### Mecanismos de Consulta Histórica
* **TIMESTAMP AS OF**: Ejecuta consultas sobre el estado de la tabla en un momento específico del tiempo para auditoría o comparación de datos.
* **VERSION AS OF**: Permite acceder a una versión específica de la tabla utilizando su número secuencial en el log de transacciones.

#### Recuperación de Datos
* **RESTORE TABLE**: Revierte la tabla a un estado anterior de forma atómica, restableciendo la metadata y los archivos correspondientes a esa versión.
* **Advertencia de Retención** <br>
Interdependencia: Ejecutar VACUUM con retención corta elimina los archivos físicos necesarios para realizar un RESTORE o consultas de Time Travel hacia versiones purgadas.

In [0]:
%sql
CREATE TABLE workspace.prueba.nyctaxi_prueba
USING DELTA
AS
SELECT *
FROM csv.`/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-01.csv.gz`

In [0]:
%sql
DESCRIBE DETAIL workspace.prueba.nyctaxi_prueba

## Análisis inicial de la tabla Delta

La tabla fue creada en formato Delta, lo que permite utilizar funcionalidades avanzadas como Time Travel, OPTIMIZE y ZORDER.

###  Estado actual
- La tabla contiene **1 solo archivo (~100 MB)** (`numFiles = 1`)
- No tiene particiones ni clustering definidos

Esto indica que los datos ya están almacenados de forma eficiente y **no existe el problema de "small files"**.

###  Configuración y features
- Compresión: `ZSTD` (optimizada para rendimiento y tamaño)
- `deletionVectors` habilitado (permite borrar datos sin reescribir archivos)

###  Conclusión
La tabla se encuentra en un estado ya optimizado, por lo que para demostrar el impacto de OPTIMIZE y ZORDER será necesario simular un escenario con múltiples archivos pequeños.
 

## Simulación de ingesta incremental

Para recrear un escenario real, los datos se insertan en múltiples iteraciones usando pequeñas muestras del dataset.

Cada escritura agrega nuevos archivos a la tabla, generando fragmentación (small files).

Este patrón puede degradar el rendimiento y requiere optimización posterior.


In [0]:
df = spark.table("workspace.prueba.nyctaxi_prueba")

In [0]:

for i in range(20):
    df.sample(fraction=0.05).write.format("delta").mode("append").saveAsTable("workspace.prueba.nyctaxi_prueba")

#mode("append") no sobrescribe, agrega datos nuevos

In [0]:
%sql
DESCRIBE DETAIL workspace.prueba.nyctaxi_prueba

##  Análisis luego de la ingesta incremental

Para simular un caso más realista, se realizó una **ingesta incremental** utilizando múltiples escrituras en modo `append`. Esto generó un escenario típico de producción donde los datos llegan en pequeños lotes.

###  Cambios en el estado de la tabla

- La tabla ahora contiene **43 archivos (~290 MB)** (`numFiles = 43`)
- Antes tenía **1 solo archivo (~100 MB)`
- No hay particiones ni clustering definidos

Esto evidencia la aparición del problema conocido como **"small files"**.

###  ¿Qué implica esto?

- Mayor cantidad de archivos → más operaciones de lectura (I/O)
- Incremento en el overhead del metadata
- Posible degradación en el rendimiento de las consultas

Este escenario es ideal para aplicar optimizaciones como:

- `OPTIMIZE` → compacta archivos pequeños en menos archivos grandes
- `ZORDER` → mejora la localización de datos en consultas con filtros

###  Configuración (sin cambios relevantes)

La tabla mantiene las mismas características modernas:

- Compresión: `ZSTD`
- `deletionVectors` habilitado

###  Conclusión

La tabla pasó de un estado **óptimo (1 archivo)** a un escenario más realista pero **subóptimo (43 archivos pequeños)** debido a la ingesta incremental.

Esto permite demostrar claramente el impacto de:

- `OPTIMIZE` → reducción de archivos y mejora en performance
- `ZORDER` → optimización para consultas selectivas

## (compactación de archivos)

###  Estado antes de OPTIMIZE

- Archivos: **43**
- Tamaño total: **~290 MB**
- Sin particiones ni clustering

Esto indica un escenario con **muchos archivos pequeños (small files)**.


In [0]:
%sql
optimize workspace.prueba.nyctaxi_prueba zorder by (_c1)

In [0]:
%sql
describe detaIl workspace.prueba.nyctaxi_prueba

##  Comparación: antes vs después de OPTIMIZE

| Métrica        | Antes de OPTIMIZE | Después de OPTIMIZE |
|----------------|------------------|---------------------|
| Archivos       | 43               | 4                   |
| Tamaño total   | 290,641,208 bytes (~290 MB) | 294,454,740 bytes (~294 MB) |
| Última modificación | 2026-05-05 16:36:32 | 2026-05-05 17:15:24 |

---

###  Análisis

- La cantidad de archivos se redujo drásticamente (**43 → 4**)
- El tamaño total se mantiene prácticamente igual (ligeras variaciones por reorganización interna)
- Se eliminaron los *small files*, consolidando los datos en archivos más grandes

---

###  Conclusión

`OPTIMIZE` logró:

- Mejorar la eficiencia de almacenamiento  
- Reducir el overhead de lectura  
- Preparar la tabla para consultas más rápidas  

---

##  ZORDER (optimización por columna)

Se aplica `ZORDER` para mejorar el rendimiento de consultas con filtros.  
Ambas operaciones pueden ejecutarse en una misma instrucción.
##  Buenas Prácticas para ZORDER

El **Z-Ordering** es una técnica de clustering que permite agrupar información relacionada en los mismos archivos. Para maximizar su impacto, sigue estas recomendaciones:

###  Cuándo utilizar ZORDER
Se recomienda aplicar Z-Ordering en columnas que cumplan con los siguientes criterios:

*   **Columnas de alta cardinalidad** 
*   **Filtros frecuentes (`WHERE`)** 
*   **Consultas analíticas**


In [0]:
%sql
vacuum workspace.prueba.nyctaxi_prueba

## DESCRIBE HISTORY

In [0]:
%sql
describe history workspace.prueba.nyctaxi_prueba

En la salida de `DESCRIBE HISTORY` podemos observar múltiples operaciones `OPTIMIZE` registradas en la columna operation, ya que Databricks puede ejecutar este tipo de optimizaciones automáticamente.

Además, las columnas **timestamp** y **version** permiten acceder a versiones anteriores de la tabla mediante las capacidades de Time Travel de Delta Lake. A continuación, veremos un ejemplo práctico, para la demostracion vamos a borrar primeramente algunos registros.


In [0]:
df.count()

In [0]:
%sql
delete from workspace.prueba.nyctaxi_prueba where _c1 between "2018-01-01" and "2019-01-01"

In [0]:
df.count()

El primer count antes de hacer el delete teniamos 20345770 y despues del delete tenemos 20344800.
Ahora lo que vamos a hacer sera volver a la version de antes del delete que nos figura en la tabla del DESCRIBE HISTORY

In [0]:
%sql
describe history workspace.prueba.nyctaxi_prueba

Volvemos a la version 26, tambien podemos volver con la fecha utilizando `TIMESTAMP AS OF` 
Con el siguiente bloque de codigo podemos ver que tenemos los registros que habiamos borrado que son 20345770

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.prueba.nyctaxi_prueba VERSION AS OF 26;

Ahora por ultimo restauramos en la utilma version de fila y hacemops 

In [0]:
spark.sql(SELECT COUNT(*)
FROM workspace.prueba.nyctaxi_prueba VERSION AS OF 26;)